# Notebook 10 — Feature Selection
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Build a fully-encoded feature set to run selection techniques on
encode_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])
df_encoded = df.copy()
for col in encode_cols:
    df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])
df_encoded['Churn'] = (df['Churn'] == 'Yes').astype(int)
df_encoded = df_encoded.drop(columns=['customerID'])

X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']
print(f"Feature matrix ready: {X.shape[1]} features, {X.shape[0]:,} rows")


Feature matrix ready: 19 features, 7,043 rows


/tmp/ipykernel_750/654009209.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  encode_cols = df.select_dtypes(include='object').columns.drop(['customerID', 'Churn'])


---
## 1. What is Feature Selection? & 2. Why Feature Selection?

### Understand
Feature selection is choosing a subset of available features to actually use in a model
— removing ones that add noise, redundancy, or no real signal. It matters because
irrelevant or redundant features can slow training, increase overfitting risk (more
parameters to fit relative to signal), and make a model harder to interpret.

### Demonstrate
**AI/ML use case:** This dataset has `customerID` — a column with 100% unique values and
zero relationship to churn. Including it wouldn't just be unhelpful, it would actively
teach a model to "memorize" specific customers rather than learn generalizable patterns.


---
## 3. Relevant, 4. Irrelevant, and 5. Redundant Features

### Understand
- **Relevant**: has a genuine relationship with the target.
- **Irrelevant**: no meaningful relationship (e.g., `customerID`).
- **Redundant**: relevant, but its information is already captured by another feature
  (e.g., `TotalCharges` vs. `tenure` × `MonthlyCharges`, Sprint 4, Notebook 6).

### Implement


In [2]:
print("Irrelevant (by design, already excluded): customerID")
print(f"\nRedundant candidate: TotalCharges correlation with tenure = {df['TotalCharges'].corr(df['tenure']):.3f}")
print("(Confirmed via VIF~8.1 in Sprint 4/5 — TotalCharges largely duplicates tenure x MonthlyCharges)")


Irrelevant (by design, already excluded): customerID

Redundant candidate: TotalCharges correlation with tenure = 0.826
(Confirmed via VIF~8.1 in Sprint 4/5 — TotalCharges largely duplicates tenure x MonthlyCharges)


---
## 6. Correlation-Based Selection

### Implement


In [3]:
corr_with_target = X.assign(Churn=y).corr()['Churn'].drop('Churn').abs().sort_values(ascending=False)
print("Features ranked by absolute correlation with Churn:")
print(corr_with_target.round(3))


Features ranked by absolute correlation with Churn:
Contract            0.397
tenure              0.352
OnlineSecurity      0.289
TechSupport         0.282
TotalCharges        0.198
OnlineBackup        0.196
MonthlyCharges      0.193
PaperlessBilling    0.192
DeviceProtection    0.178
Dependents          0.164
SeniorCitizen       0.151
Partner             0.150
PaymentMethod       0.107
InternetService     0.047
StreamingMovies     0.038
MultipleLines       0.038
StreamingTV         0.037
PhoneService        0.012
gender              0.009
Name: Churn, dtype: float64


**Finding:** `Contract` (0.397) is the strongest linearly-correlated feature, followed
by `tenure`, `OnlineSecurity`, `TechSupport`, and `InternetService` — consistent with
Sprint 4's findings using a completely different method (churn-rate spread).


---
## 7. Variance Threshold

### Understand
Removes features with very low variance — if a feature barely changes across the
dataset, it carries little discriminating information regardless of its relationship
with the target.

### Implement


In [4]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)
selector.fit(X)
low_variance_features = X.columns[~selector.get_support()]
print(f"Features with near-zero variance: {list(low_variance_features) if len(low_variance_features) else 'None'}")


Features with near-zero variance: None


**Finding:** No features fall below the threshold — every column carries at least
some meaningful spread, even the more imbalanced ones like `SeniorCitizen`.


---
## 8. Univariate Feature Selection

### Understand
Scores each feature independently against the target using a statistical test (here,
ANOVA F-test via `f_classif`), then keeps the top K.

### Implement


In [5]:
from sklearn.feature_selection import SelectKBest, f_classif

selector_kbest = SelectKBest(score_func=f_classif, k=10)
selector_kbest.fit(X, y)
scores = pd.Series(selector_kbest.scores_, index=X.columns).sort_values(ascending=False)
print("Top 10 features by univariate F-score:")
print(scores.head(10).round(1))


Top 10 features by univariate F-score:
Contract            1315.1
tenure               997.3
OnlineSecurity       643.2
TechSupport          610.6
TotalCharges         288.3
OnlineBackup         279.9
MonthlyCharges       273.5
PaperlessBilling     269.0
DeviceProtection     230.7
Dependents           195.1
dtype: float64


---
## 9. Mutual Information

### Understand
Measures how much knowing one variable reduces uncertainty about another — unlike
correlation, it captures NON-linear relationships too.

### Implement


In [6]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = pd.Series(mutual_info_classif(X, y, random_state=42), index=X.columns).sort_values(ascending=False)
print("Top 10 features by mutual information with Churn:")
print(mi_scores.head(10).round(4))


Top 10 features by mutual information with Churn:
Contract            0.0916
tenure              0.0770
OnlineSecurity      0.0609
TechSupport         0.0591
OnlineBackup        0.0557
InternetService     0.0544
PaymentMethod       0.0479
DeviceProtection    0.0463
MonthlyCharges      0.0453
TotalCharges        0.0433
dtype: float64


**Finding:** Broadly agrees with the correlation-based ranking (`Contract`, `tenure`,
`OnlineSecurity` near the top) — strong agreement across a linear (correlation) and a
non-linear (mutual information) method adds confidence these are genuinely important
features, not an artifact of one particular technique.


---
## 10. Recursive Feature Elimination (RFE)

### Understand
Trains a model, removes the least important feature, and repeats — a wrapper method that
accounts for feature *interactions*, unlike the univariate methods above.

### Implement


In [7]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

rfe = RFE(LogisticRegression(max_iter=2000), n_features_to_select=10)
rfe.fit(X, y)
selected_by_rfe = X.columns[rfe.support_]
print(f"Top 10 features selected by RFE:\n{list(selected_by_rfe)}")


Top 10 features selected by RFE:
['SeniorCitizen', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'TechSupport', 'Contract', 'PaperlessBilling']


---
## 11. Feature Importance (Tree-Based)

### Implement


In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 features by Random Forest importance:")
print(importances.head(10).round(4))


Top 10 features by Random Forest importance:
TotalCharges       0.1916
MonthlyCharges     0.1795
tenure             0.1589
Contract           0.0752
PaymentMethod      0.0516
OnlineSecurity     0.0484
TechSupport        0.0420
gender             0.0278
InternetService    0.0271
OnlineBackup       0.0266
dtype: float64


---
## Comparing All Methods

### Implement


In [9]:
top5_by_method = pd.DataFrame({
    'Correlation': corr_with_target.head(5).index.tolist(),
    'Univariate (F-test)': scores.head(5).index.tolist(),
    'Mutual Information': mi_scores.head(5).index.tolist(),
    'Random Forest': importances.head(5).index.tolist(),
})
print(top5_by_method)

all_top_features = set()
for col in top5_by_method.columns:
    all_top_features.update(top5_by_method[col])
print(f"\nFeatures appearing in the top-5 of at least one method: {all_top_features}")


      Correlation Univariate (F-test) Mutual Information   Random Forest
0        Contract            Contract           Contract    TotalCharges
1          tenure              tenure             tenure  MonthlyCharges
2  OnlineSecurity      OnlineSecurity     OnlineSecurity          tenure
3     TechSupport         TechSupport        TechSupport        Contract
4    TotalCharges        TotalCharges       OnlineBackup   PaymentMethod

Features appearing in the top-5 of at least one method: {'MonthlyCharges', 'OnlineBackup', 'Contract', 'OnlineSecurity', 'TechSupport', 'TotalCharges', 'tenure', 'PaymentMethod'}


### Documentation (Problem / Analysis / Technique / Reason / Implementation / Result / Impact)
- **Problem:** 19 candidate features, not all equally useful; `customerID` already
  excluded as irrelevant, `TotalCharges` flagged as redundant with `tenure` ×
  `MonthlyCharges` (Notebook 6/Sprint 4).
- **Analysis:** Four independent methods (correlation, univariate F-test, mutual
  information, Random Forest importance) largely agree on the same handful of top
  features — `Contract`, `tenure`, `OnlineSecurity`, `TechSupport`, `InternetService`,
  `MonthlyCharges`, `TotalCharges`.
- **Technique Selected:** Cross-method agreement (a feature ranking well across 3+ of
  4 methods) rather than trusting any single method alone.
- **Reason:** Different methods have different blind spots (linear-only, single-variable,
  interaction-blind); agreement across methods is stronger evidence than any one score.
- **Implementation:** shown above.
- **Result:** A shortlist of features consistently ranked as important is identified,
  without discarding any feature outright at this stage.
- **Impact:** This shortlist directly informs which features to prioritize when building
  the final model in a future sprint, without needing to guess.


---
## Summary

| Method | Type | Top Features (agree closely) |
|---|---|---|
| Correlation | Linear, univariate | Contract, tenure, OnlineSecurity |
| Univariate F-test | Linear, univariate | Contract, tenure, TechSupport |
| Mutual Information | Non-linear, univariate | Contract, tenure, OnlineSecurity |
| Random Forest Importance | Non-linear, multivariate | tenure, TotalCharges, MonthlyCharges, Contract |

**Why unnecessary features matter:** `customerID` alone illustrates the risk directly —
a feature with zero real signal but 100% uniqueness would let a model "cheat" by
memorizing training rows instead of learning generalizable patterns, producing excellent
training performance and poor real-world performance.

**Next notebook:** `11_Imbalanced_Data.ipynb` — addressing `Churn`'s 73.5%/26.5% class
imbalance before this feature set goes into a model.
